In [ ]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))


In [ ]:
pip install transformers

In [ ]:
import os
hf_token = os.getenv("HF_TOKEN")

In [ ]:
import os

In [ ]:
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name = "google/gemma-3-1b-it"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
tokenizer("Hello Parth")

In [ ]:
input_conversation = [
    {"role" : "user", "content" : "Who is the most Powerfull man in the world ?"},
    {"role" : "assistant", "content" : "The most Powerfull man in the world is"}
]

In [ ]:
input_tokens = tokenizer.apply_chat_template(
    conversation = input_conversation,
    tokenize = True
).to(device)
input_tokens

In [ ]:
input_detokens = tokenizer.apply_chat_template(
    conversation = input_conversation,
    tokenize = False,
    continue_final_message = True
)
input_detokens

In [ ]:
output_label = "only and only The Parth Maheshwari from Ajmer"
full_conversation = input_detokens + output_label + tokenizer.eos_token
full_conversation

In [ ]:
input_tokenized = tokenizer(full_conversation, return_tensors="pt", add_special_tokens=False).to(device)["input_ids"]
input_tokenized

In [ ]:
input_ids = input_tokenized[:,:-1].to(device)
target_ids = input_tokenized[:,1:].to(device)
target_ids

Function to Calculate the Loss

In [54]:
import torch.nn as nn
def calculate_loss(logits, labels):
    loss_fn = nn.CrossEntropyLoss(reduction="none")
    cross_entropy = loss_fn(logits.view(-1, logits.shape[-1]),labels.view(-1))
    return cross_entropy

In [ ]:
import torch
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype = torch.bfloat16
).to(device)

In [56]:
from torch.optim import AdamW
import torch

device = "cuda"
model = model.to(device)

scaler = torch.cuda.amp.GradScaler()

optimizer = AdamW(
    model.parameters(),
    lr=3e-5,
    weight_decay=0.01
)

model.train()

for _ in range(10):

    optimizer.zero_grad()

    with torch.cuda.amp.autocast():
        out = model(input_ids=input_ids)
        loss = calculate_loss(out.logits, target_ids).mean()

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    torch.cuda.empty_cache()


C:\Users\HP\AppData\Local\Temp\ipykernel_18976\174446048.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
C:\Users\HP\AppData\Local\Temp\ipykernel_18976\174446048.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
input_prompts = "Who is the most Powerfull man in the world ?"
i_tokens = tokenizer(input_prompts, return_tensors="pt")["input_ids"].to(device)

output_tokens = model.generate(i_tokens)
output_tokens
de_tokenized_output = tokenizer.batch_decode(output_tokens)
de_tokenized_output